# ROMP -> normalized SMPL `.npz`

Runs [ROMP](https://github.com/Arthur151/ROMP) (per-frame SMPL from one monocular clip)
and writes `<video_id>.smpl.npz`, which `tools/smpl_to_skeleton.py --wham-output` turns
into `skeleton.json v2` and feeds the Blender SMPL mesh (Route A).

**Engine note:** we switched from WHAM to ROMP. WHAM's Colab install (conda + a compiled
SLAM module + checkpoint scripts) is very painful; ROMP is a pip package that outputs the
same SMPL params (θ, betas). WHAM / 4D-Humans stay as later quality upgrades — the npz
contract is identical, so nothing downstream changes.

- Design: `docs/superpowers/specs/2026-07-23-monocular-smpl-skeleton-design.md`
- Output contract (npz keys): see `tools/colab/README.md`.

**One-time setup:** Runtime -> Change runtime type -> GPU. Have your neutral SMPL model
ready to upload: locally it is `models/smpl/SMPL_NEUTRAL.pkl`. Then run the cells top to bottom.


In [ ]:
# ROMP is a plain pip package (no repo clone, no compile).
!pip install -q simple_romp==1.1.4
import romp
print("simple_romp installed OK")
# If you hit a numpy error (ROMP uses old np aliases), run in a fresh cell:
#   !pip install -q "numpy<2"
# then Runtime -> Restart session, and re-run from here.


In [ ]:
# One-time SMPL prep for ROMP.
# The official SMPL_NEUTRAL.pkl stores chumpy objects, and chumpy is broken on Colab's
# Python 3.12 (inspect.getargspec removed) + numpy 2 (np.bool/np.float removed). We patch
# just enough to import chumpy HERE, materialize its arrays to plain numpy, and save a
# CLEAN pkl. ROMP's prepare_smpl (a separate process) then loads the clean pkl and never
# needs chumpy. You also get SMPL_NEUTRAL_clean.pkl back to reuse next time.
import os, glob, shutil, pickle
!pip install -q chumpy

import numpy as np, inspect
for n, v in {"bool": np.bool_, "int": np.int_, "float": np.float64, "complex": np.complex128,
             "object": np.object_, "str": np.str_, "unicode": np.str_}.items():
    if not hasattr(np, n):
        setattr(np, n, v)
np.nan, np.inf = float("nan"), float("inf")
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec          # removed in Py3.11; chumpy still calls it
import chumpy  # noqa: needed so pickle can reconstruct the chumpy arrays
print("chumpy imported OK")

!wget -q https://github.com/Arthur151/ROMP/releases/download/V2.0/smpl_model_data.zip
!unzip -o -q smpl_model_data.zip -d smpl_model_data

from google.colab import files
print("Upload SMPL_NEUTRAL.pkl  (your local models/smpl/SMPL_NEUTRAL.pkl)")
up = files.upload()
src_pkl = next(iter(up))

with open(src_pkl, "rb") as f:
    m = pickle.load(f, encoding="latin1")
# Densify only the chumpy entries; leave numpy / scipy-sparse entries untouched.
clean = {k: (np.array(v) if type(v).__module__.split(".")[0] == "chumpy" else v)
         for k, v in m.items()}

meta_dir = os.path.dirname(glob.glob("smpl_model_data/**/J_regressor_extra.npy", recursive=True)[0])
clean_pkl = os.path.join(meta_dir, "SMPL_NEUTRAL.pkl")
with open(clean_pkl, "wb") as f:
    pickle.dump(clean, f)
print("de-chumpified ->", meta_dir, "| contents:", sorted(os.listdir(meta_dir)))

# Convert to ROMP's ~/.romp/SMPL_NEUTRAL.pth (clean pkl -> no chumpy in the subprocess).
!romp.prepare_smpl -source_dir="{meta_dir}"
romp_dir = os.path.expanduser("~/.romp")
ok = os.path.exists(os.path.join(romp_dir, "SMPL_NEUTRAL.pth"))
print("converted ->", ok, "| ~/.romp:", os.listdir(romp_dir) if os.path.isdir(romp_dir) else "MISSING")

# Hand back the clean model so future runs skip chumpy entirely (upload THIS next time).
if ok:
    shutil.copy(clean_pkl, "SMPL_NEUTRAL_clean.pkl")
    files.download("SMPL_NEUTRAL_clean.pkl")
    print("Downloaded SMPL_NEUTRAL_clean.pkl — keep it beside models/smpl/ for reuse.")


In [ ]:
VIDEO_ID = "test_6"                      # test_6 = the Pexels person-moving clip
up = files.upload()                      # pick your .mp4 (e.g. test_6.mp4)
clip = next(iter(up))

# ROMP does not need 4K -> downscale to 720p (fps unchanged) for speed.
!ffmpeg -y -loglevel error -i "{clip}" -vf "scale=-2:720" -an input_720.mp4

# --calc_smpl saves the SMPL params to an .npz. We SKIP --render_mesh/--save_video:
# headless Colab has no display and the mesh renderer often crashes there; we only need the data.
!mkdir -p out_romp
!romp --mode=video --calc_smpl -i=input_720.mp4 -o=out_romp/{VIDEO_ID}.mp4

import glob
print("npz produced:", glob.glob("out_romp/**/*.npz", recursive=True))


In [ ]:
import numpy as np, glob, os, cv2
from google.colab import files
!pip install -q smplx

# ROMP's combined video output: a dict keyed by frame filename.
res = np.load("out_romp/video_results.npz", allow_pickle=True)["results"][()]
frames = sorted(res.keys())

def person0(fr):
    if isinstance(fr, np.ndarray): fr = fr.tolist()
    if isinstance(fr, dict): return fr
    if isinstance(fr, (list, tuple)) and len(fr): return fr[0]
    return {}

d0 = person0(res[frames[0]])
print(len(frames), "frames; per-frame keys:", list(d0.keys()))

pose, betas_l, transl = [], [], []
for k in frames:
    d = person0(res[k])
    if "smpl_thetas" not in d or not np.asarray(d["smpl_thetas"]).size:
        continue                                       # frame with no detected person
    th = np.asarray(d["smpl_thetas"], np.float32)
    be = np.asarray(d["smpl_betas"],  np.float32)
    ct = np.asarray(d["cam_trans"] if "cam_trans" in d else d["cam"], np.float32)
    if th.ndim == 2:                                   # (N persons, ...) -> person 0
        th, be, ct = th[0], be[0], ct[0]
    pose.append(th[:72]); betas_l.append(be[:10]); transl.append(ct[:3])

pose   = np.asarray(pose, np.float32)                  # (T,72)
betas  = np.asarray(betas_l, np.float32).mean(0)       # (10,)
transl = np.asarray(transl, np.float32)                # (T,3)
print("kept", len(pose), "frames")

# ROMP doesn't save 3D joints -> regress the 24 SMPL joints from (theta, betas).
import smplx, torch, shutil
os.makedirs("body_models/smpl", exist_ok=True)
shutil.copy(glob.glob("smpl_model_data/**/SMPL_NEUTRAL.pkl", recursive=True)[0],
            "body_models/smpl/SMPL_NEUTRAL.pkl")
body = smplx.create("body_models", model_type="smpl", gender="neutral", batch_size=len(pose))
out  = body(global_orient=torch.tensor(pose[:, :3]),
            body_pose=torch.tensor(pose[:, 3:72]),
            betas=torch.tensor(np.repeat(betas[None], len(pose), 0)),
            transl=torch.tensor(transl))
joints3d = out.joints.detach().cpu().numpy()[:, :24, :].astype(np.float32)   # (T,24,3)

fps = cv2.VideoCapture("input_720.mp4").get(cv2.CAP_PROP_FPS) or 30.0
np.savez(f"{VIDEO_ID}.smpl.npz", joints3d=joints3d, pose=pose,
         betas=betas, transl=transl, fps=np.array(fps))
print("shapes:", joints3d.shape, pose.shape, betas.shape, transl.shape, "| fps", fps)
files.download(f"{VIDEO_ID}.smpl.npz")
